# Fábrica de Corpus: Radar Cidadão

Disciplina de NLP, Global Solution 2026.2 (FIAP, CitizenTech). Este notebook transforma documentos públicos sobre chuvas intensas, alagamentos e deslizamentos em três corpus JSONL (`corpus_texto`, `corpus_qa` e `corpus_chat`) e abre uma interface Gradio para testar a ferramenta.

**Como rodar no Google Colab**

1. Abra este notebook no Colab (Arquivo > Abrir notebook > GitHub, ou pelo link do README).
2. O ambiente **sem GPU** basta: a geração com o Qwen é opcional e está desligada (etapa 5).
3. Execute as células em ordem (Ambiente de execução > Executar tudo). A primeira clona o repositório e instala as bibliotecas; leva alguns minutos.
4. Na última célula aparece um link público do Gradio (`share=True`) para usar a interface.

Fluxo da ferramenta: ingestão -> extração com a fonte -> limpeza e normalização -> anonimização -> segmentação (256 a 512 tokens) -> geração com o Qwen (opcional, GPU) -> revisão humana -> finalização e relatório de qualidade.

In [1]:
# 0. Preparação: no Colab, clona o repositório e instala as bibliotecas; fora dele, usa a pasta local
import subprocess, sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    REPO = Path("/content/radar-cidadao")
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/sarahmfurlan-sudo/radar-cidadao.git", str(REPO)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
    PROJETO = REPO / "nlp-fabrica-corpus"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJETO / "requirements.txt")], check=True)
else:
    PROJETO = Path.cwd() if (Path.cwd() / "src" / "fabrica_corpus").exists() else Path.cwd() / "nlp-fabrica-corpus"

if str(PROJETO / "src") not in sys.path:
    sys.path.insert(0, str(PROJETO / "src"))
commit = subprocess.run(["git", "-C", str(PROJETO), "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Projeto:", PROJETO, "| commit:", commit or "(sem git)", "| Colab:", EM_COLAB)


def mostrar(texto_md):
    """Mostra Markdown formatado no notebook (ou texto puro fora dele)."""
    try:
        from IPython.display import Markdown, display
        display(Markdown(texto_md))
    except ImportError:
        print(texto_md)

Projeto: /content/radar-cidadao/nlp-fabrica-corpus | commit: 61fb0c5 | Colab: True


## 1 e 2. Ingestão e extração com a fonte

Lê os 5 documentos de `dados/entrada` (PDF, HTML) aplicando a seleção de páginas de `selecao_paginas.json` (capa, créditos, sumário e referências ficam de fora). Cada trecho guarda o arquivo de origem e a página.

In [2]:
from collections import Counter
from fabrica_corpus.exportacao import ENTRADA
from fabrica_corpus.ingestao import ingerir_pasta

trechos = ingerir_pasta(ENTRADA)
por_fonte = Counter(t.fonte for t in trechos)
print(f"{len(trechos)} trechos extraídos de {len(por_fonte)} documentos:")
for fonte, n in por_fonte.items():
    print(f"  {fonte}: {n} trecho(s)")
t = trechos[0]
print(f"\nExemplo: fonte={t.fonte} | tipo={t.tipo} | página={t.pagina}\n{t.texto[:400]}")

196 trechos extraídos de 5 documentos:
  mcid_2024_periferia_sem_risco.pdf: 7 trecho(s)
  mcid_2026_mapeamento_riscos.pdf: 102 trecho(s)
  ms_2017_cartilha_alerta_chuvas.pdf: 19 trecho(s)
  ms_2025_plano_contingencia_chuvas.pdf: 67 trecho(s)
  ms_2026_enchentes.html: 1 trecho(s)

Exemplo: fonte=mcid_2024_periferia_sem_risco.pdf | tipo=pdf | página=3
T odos os anos, comunidades e periferias sofrem muito com 
desastres relacionados a excesso de chuvas no Brasil. 
O Governo Federal ajuda a prefeitura da sua cidade a proteger 
a população local financiando obras e ações preventivas, mas 
algumas delas podem demorar para ficar prontas. 
Como a proteção das pessoa é urgente e não pode esperar, 
podemos começar a agir já. 
Vamos juntos?
Que tal
come


## 3 e 4. Limpeza e normalização

Unicode e aspas, cabeçalhos e rodapés repetidos, números de página, sumário, rodapé de página web, hifenização e letra capitular, linhas quebradas, blocos de referências bibliográficas e espaços. Acentos, pontuação e maiúsculas são mantidos; não há remoção de stop words nem stemming.

In [3]:
from fabrica_corpus.limpeza import limpar_corpus

limpos, est_limpeza = limpar_corpus(trechos)
print("Caracteres antes:", est_limpeza["caracteres_antes"], "| depois:", est_limpeza["caracteres_depois"])
print("Trechos vazios descartados:", est_limpeza["trechos_vazios_descartados"])
print("Linhas de referência removidas:", est_limpeza["linhas_de_referencia_removidas"])
print("Cabeçalhos/rodapés repetidos por documento:", est_limpeza["repetidas_por_documento"])

Caracteres antes: 579935 | depois: 528236
Trechos vazios descartados: 0
Linhas de referência removidas: 184
Cabeçalhos/rodapés repetidos por documento: {'mcid_2026_mapeamento_riscos.pdf': 1, 'ms_2017_cartilha_alerta_chuvas.pdf': 2, 'ms_2025_plano_contingencia_chuvas.pdf': 2}


## 5. Anonimização

CPF, e-mail, telefone, nomes de pessoas (spaCy `pt_core_news_md` + regras) e sobrenomes de autores em citações viram `[CPF]`, `[EMAIL]`, `[TELEFONE]` e `[NOME]`. Números de emergência (190, 192, 193, 199, 40199) e o 156 são preservados. Abaixo, o "antes e depois" no arquivo de teste com **dados fictícios** e as contagens no corpus real.

In [4]:
from fabrica_corpus.anonimizacao import anonimizar_corpus
from fabrica_corpus.ingestao import extrair_arquivo

teste, _ = limpar_corpus(extrair_arquivo(PROJETO / "dados" / "teste" / "teste_dados_ficticios.txt"))
teste_anon, cont_teste = anonimizar_corpus(teste)
print("=== ANTES (dados fictícios) ===\n" + teste[0].texto)
print("\n=== DEPOIS ===\n" + teste_anon[0].texto)
print("\nContagem no arquivo de teste:", cont_teste)

anonimos, est_anon = anonimizar_corpus(limpos)
print("Contagem no corpus real (5 documentos):", est_anon)

=== ANTES (dados fictícios) ===
DOCUMENTO DE TESTE - TODOS OS DADOS PESSOAIS SÃO FICTÍCIOS

Registro de ocorrências de chuva - Associação de Moradores da Vila Esperança

Na madrugada de 12 de janeiro de 2026, a moradora Mariana Toledo Vasconcelos registrou no Radar Cidadão um alagamento na esquina da Rua das Palmeiras com a Rua do Córrego. Segundo ela, a água chegou a 40 centímetros em menos de uma hora. O vizinho Joaquim Pereira Lins confirmou o relato e enviou fotos pelo aplicativo. Mariana também relatou que o bueiro da esquina estava entupido havia duas semanas.

Contato da moradora: CPF 123.456.789-10, e-mail mariana.toledo@example.com, telefone (20) 99999-0001.
Contato do vizinho: CPF 987.654.321-99, e-mail joaquim.lins@example.com, celular +55 20 98888-0002.

A Defesa Civil foi acionada pelo telefone 199 e orientou os moradores a se afastarem das áreas de encosta. Dona Benedita Arruda, de 78 anos, precisou de ajuda para sair de casa. O Corpo de Bombeiros (193) fez o resgate, e o

## 6. Segmentação

Trechos de 256 a 512 tokens, contados com o tokenizador do Qwen 2.5 (o mesmo vocabulário do modelo gerador), sempre cortando em fim de frase. Cada segmento recebe um `segmento_id` (arquivo + número) e guarda as páginas de origem. Na primeira execução o tokenizador é baixado do Hugging Face.

In [5]:
from fabrica_corpus.segmentacao import MODELO_TOKENIZADOR, segmentar

segmentos, est_seg = segmentar(anonimos)
print("Tokenizador:", MODELO_TOKENIZADOR)
print({k: est_seg[k] for k in ("segmentos", "tokens_media", "tokens_min", "tokens_max", "segmentos_curtos")})
print("Segmentos por documento:", est_seg["segmentos_por_fonte"])
s = segmentos[0]
print(f"\n{s.id} | páginas {s.paginas} | {s.tokens} tokens\n{s.texto[:500]}")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Tokenizador: Qwen/Qwen2.5-7B-Instruct
{'segmentos': 307, 'tokens_media': 480.6, 'tokens_min': 24, 'tokens_max': 512, 'segmentos_curtos': 1}
Segmentos por documento: {'mcid_2024_periferia_sem_risco.pdf': 4, 'mcid_2026_mapeamento_riscos.pdf': 203, 'ms_2017_cartilha_alerta_chuvas.pdf': 12, 'ms_2025_plano_contingencia_chuvas.pdf': 86, 'ms_2026_enchentes.html': 2}

mcid_2024_periferia_sem_risco_0001 | páginas [3, 4, 5] | 507 tokens
Todos os anos, comunidades e periferias sofrem muito com desastres relacionados a excesso de chuvas no Brasil. O Governo Federal ajuda a prefeitura da sua cidade a proteger a população local financiando obras e ações preventivas, mas algumas delas podem demorar para ficar prontas. Como a proteção das pessoa é urgente e não pode esperar, podemos começar a agir já. Vamos juntos? Que tal começar a gestão participativa de riscos de desastres onde você mora? Proteger a população é uma tarefa coletiva


## 7. Geração das perguntas e dos diálogos (opcional, precisa de GPU)

As perguntas (`corpus_qa`) e os diálogos (`corpus_chat`) foram gerados com o **Qwen2.5-14B-Instruct** em 4 bits, no Colab com GPU, a partir dos fatos extraídos de cada segmento; passaram por travas automáticas, por um juiz de coerência e pela revisão humana de 20% da amostra. Os corpus entregues são essa versão **revisada**, em `entrega-final/`.

Esta etapa fica **desligada** (`RODAR_GERACAO = False`): gerar de novo exige GPU, leva horas e produziria exemplos sem revisão. Para demonstrar, ligue a opção num ambiente com GPU; ela gera só alguns exemplos, sem tocar na entrega.

In [6]:
from fabrica_corpus import geracao

RODAR_GERACAO = False   # mude para True só num ambiente com GPU

print("Modelo:", geracao.MODELO_GERADOR, "| prompt QA/chat:", geracao.VERSAO_QA_CHAT, "| recusas:", geracao.VERSAO_PROMPT)
if RODAR_GERACAO:
    import torch
    assert torch.cuda.is_available(), "Ative uma GPU (Ambiente de execução > Alterar o tipo de ambiente)."
    gerar = geracao.carregar_qwen()
    demo_qa, demo_est = geracao.gerar_qa(segmentos, gerar, total=4, por_segmento=2)
    for e in demo_qa:
        print("\nP:", e["pergunta"], "\nR:", e["resposta"])
else:
    print("Geração desligada: os corpus usados a seguir são os da versão revisada (entrega-final).")

Modelo: Qwen/Qwen2.5-14B-Instruct | prompt QA/chat: v8_3 | recusas: v8_2
Geração desligada: os corpus usados a seguir são os da versão revisada (entrega-final).


## 8. Finalização e relatório de qualidade

A finalização trabalha sobre os corpus **revisados**: refaz o `corpus_texto` pelo pipeline acima, tira as páginas das citações, valida campos, falas, duplicatas, quantidades mínimas e recusas válidas. Aqui ela roda numa **cópia** da `entrega-final`, só para mostrar que o resultado é reproduzido; depois vem o relatório (quantidades, média de tokens, duplicatas e 5 exemplos aleatórios de cada corpus).

In [7]:
import json, shutil, tempfile
from fabrica_corpus.finalizacao import ENTREGA, finalizar

copia = Path(tempfile.mkdtemp()) / "entrega"
shutil.copytree(ENTREGA, copia)
metricas = finalizar(copia, segmentos=segmentos)
for nome in ("corpus_texto.jsonl", "corpus_qa.jsonl", "corpus_chat.jsonl"):
    igual = (copia / nome).read_text(encoding="utf-8") == (ENTREGA / nome).read_text(encoding="utf-8").replace("\r\n", "\n")
    print(f"{nome}: reproduzido igual à entrega? {igual}")
print("Recusas válidas:", metricas["recusas_validas"], "| respostas com página citada:", metricas["respostas_com_paginas"])

corpus_texto.jsonl: reproduzido igual à entrega? True
corpus_qa.jsonl: reproduzido igual à entrega? True
corpus_chat.jsonl: reproduzido igual à entrega? True
Recusas válidas: 11 | respostas com página citada: 0


In [8]:
from fabrica_corpus.interface import exemplos_corpus, resumo_corpus

mostrar(resumo_corpus(ENTREGA))
mostrar(exemplos_corpus(ENTREGA, semente=2026))

As perguntas e os diálogos vêm da versão revisada (geração com o Qwen2.5-14B-Instruct no Colab, revisão humana e finalização). A geração com o Qwen é opcional e precisa de GPU; o notebook não gera de novo.



| Arquivo | Exemplos | Mínimo | Média de tokens (aprox.) | Duplicatas removidas |
|---|---|---|---|---|
| corpus_texto.jsonl | 307 | 50 | 326.5 | 0 |
| corpus_qa.jsonl | 100 | 100 | 93.1 | 15 |
| corpus_chat.jsonl | 31 | 30 | 306.8 | 0 |



**Recusas válidas:** 11 (mínimo 10)

**Revisão humana:** Sarah Marangoni Furlan (RM 566764); 27 exemplos (20.61%): 23 S e 2 N.

### corpus_texto

| segmento_id | Fonte | Texto |
|---|---|---|
| mcid_2026_mapeamento_riscos_0057 | mcid_2026_mapeamento_riscos.pdf | Para tanto, propõe-se que a classificação do perigo se fundamente em dois pilares: a altura da lâmina d'água e a frequência de ocorrência em um intervalo de dez… |
| mcid_2026_mapeamento_riscos_0160 | mcid_2026_mapeamento_riscos.pdf | Nos mapeamentos para deslizamentos e rastejos, quando a área possui infraestrutura instalada e as construções apresentam elevada capacidade de resistência, não … |
| ms_2025_plano_contingencia_chuvas_0039 | ms_2025_plano_contingencia_chuvas.pdf | - verificar se o evento alterou a normalidade do serviço dos laboratórios e se há necessidade de reorganizar; verificar estoque de kits para sorologia da leptos… |
| ms_2025_plano_contingencia_chuvas_0044 | ms_2025_plano_contingencia_chuvas.pdf | Realizar o monitoramento e análise epidemiológica dos casos de DDA e avaliar alteração do padrão epidemiológico de casos, internações e óbitos, fornecendo subsí… |
| mcid_2026_mapeamento_riscos_0049 | mcid_2026_mapeamento_riscos.pdf | A base de dados, também conhecida como conjunto de treinamento, é composta por dados históricos e utilizada na fase de aprendizado, onde o sistema adquire o con… |

### corpus_qa

| Pergunta (instruction) | Resposta (output) | Fonte |
|---|---|---|
| O que eu faço se sentir febre depois de entrar em água ou lama de enchente? | Procure atendimento médico. Fonte: Plano de Contingência para Chuvas Intensas, do Ministério da Saúde (2025). | ms_2025_plano_contingencia_chuvas.pdf |
| Qual é o risco de doenças depois de alagamentos e deslizamentos? | Há risco de doenças transmissíveis como leptospirose e hepatite A após alagamentos e deslizamentos. Fonte: Plano de Contingência para Chuvas Intensas, do Minist… | ms_2025_plano_contingencia_chuvas.pdf |
| O que é o índice de risco? | No método de avaliação de risco de queda de blocos, o índice de risco (IR) resulta da soma das pontuações de sinais, abrangência e consequência. A soma classifi… | mcid_2026_mapeamento_riscos.pdf |
| O que acontece em áreas nas cabeceiras dos rios? | Áreas nas cabeceiras dos rios podem ter enxurradas porque a água escorre rápido. Fonte: Mapeamento de Riscos, do Ministério das Cidades (2026). | mcid_2026_mapeamento_riscos.pdf |
| Quais órgãos são citados pelo plano para acompanhar condições climáticas e alertas de desastres? | O plano cita o Cemaden, o Instituto Nacional de Meteorologia (Inmet) e a Defesa Civil como instituições de referência para o monitoramento de condições climátic… | ms_2025_plano_contingencia_chuvas.pdf |

### corpus_chat

**recusa** | fonte `fora_do_escopo`

- **user:** Vai chover forte amanhã às 15h na minha rua?
- **assistant:** Não consigo te informar sobre previsões específicas, mas você pode acompanhar os alertas da Defesa Civil pelo SMS 40199.
- **user:** Entendi. E agora?
- **assistant:** Continue acompanhando os alertas oficiais para ficar sempre bem informado.

---

**chat** | fonte `mcid_2026_mapeamento_riscos.pdf`

- **user:** Por que minha área está sempre alagando?
- **assistant:** As inundações podem acontecer por causa de chuvas fortes que duram vários dias ou até mesmo por chuvas concentradas em pouco tempo.
- **user:** E quando a água sobe devagar, a gente precisa sair correndo?
- **assistant:** Não, inundações que acontecem devagar têm mais tempo de duração e podem afetar a vida econômica e social das pessoas, mas geralmente não requerem evacuação urgente.
- **user:** Então posso sair antes se ver que vai alagar?
- **assistant:** Sim, em casos de inundações lentas, pode-se fazer uma evacuação preventiva para evitar problemas maiores. Fonte: Mapeamento de Riscos, do Ministério das Cidades (2026).

---

**chat** | fonte `ms_2017_cartilha_alerta_chuvas.pdf`

- **user:** Quando tem alerta de chuva forte, eu tenho que tirar os meus aparelhos da tomada?
- **assistant:** Sim, retire os aparelhos eletrônicos das tomadas quando houver alerta de chuva intensa.
- **user:** E também tenho que fechar as válvulas de água e gás quando tem alerta de chuva?
- **assistant:** Sim, feche as entradas de gás e os registros de água quando houver alerta de chuva intensa.
- **user:** Tenho que pegar meus cachorros e outros animais antes da chuva?
- **assistant:** Sim, recolha os animais quando houver alerta de chuva intensa. Fonte: Cartilha de Alerta de Chuvas Intensas, do Ministério da Saúde (2017).

---

**recusa** | fonte `fora_do_escopo`

- **user:** Posso processar a prefeitura por conta dos alagamentos e quanto ganho com isso?
- **assistant:** Não consigo determinar, com esse material, se há direito a indenização nem estimar valores. Isso depende dos fatos e de avaliação jurídica do caso.
- **user:** Entendi. E agora?
- **assistant:** Se quiser avaliar seus direitos, reúna documentos e registros do ocorrido e procure orientação jurídica. Não posso estimar um valor de indenização com base no trecho disponível.

---

**recusa** | fonte `ms_2017_cartilha_alerta_chuvas.pdf`

- **user:** Apareceu uma rachadura grande na parede de casa depois da chuva. Minha casa vai cair?
- **assistant:** Não posso fazer esse tipo de avaliação. Ligue para a Defesa Civil (199) para que eles possam verificar a situação da sua casa.
- **user:** Entendi. E agora?
- **assistant:** Por favor, ligue para a Defesa Civil (199) imediatamente.

## 9. Interface Gradio

- **Processar documento:** envie PDF, TXT, DOCX ou HTML (vários arquivos) ou cole um texto e veja cada etapa: extração com a fonte, limpeza, anonimização e segmentação. No fim, baixe o `corpus_texto.jsonl` daquele envio. Para testar a anonimização, use `dados/teste/teste_dados_ficticios.docx` (dados fictícios, com tabela de contatos).
- **Corpus finais:** estatísticas, 5 exemplos aleatórios de cada corpus e download dos 3 JSONL da entrega.

No Colab, `share=True` gera um link público temporário.

In [9]:
from fabrica_corpus.interface import criar_app

app = criar_app(ENTREGA)
if EM_COLAB:
    app.launch(share=True)
else:
    print("Fora do Colab: rode app.launch() para abrir a interface localmente.")

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e4ea3ecb4b1767e8b9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
